In [9]:
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b") 

In [10]:
#tool 
from langchain_core.tools import tool

@tool #-> a decorator whoeer you assign it to it becomes that and it convert py func to langhain tool object!
def get_weather(city:str)->str:
    """Get the weather of the city."""
    return f"The weather in {city} is chilly!"

model_with_tools=model.bind_tools([get_weather])

#tools rovde parimgof schema including name of db,websearch ,tool call and all
where langchai need to interac with api,db.custom logic to fulfill req....

"""
@tool: Defines a single capability. It converts a Python function into a Tool object that an agent can use.  It specifies the name, description, and input schema for one specific action (e.g., get_weather, calculate_tax).


create_agent: Builds the entire agent system.  It is a factory function that constructs the reasoning loop, state management, and execution logic. It consumes a list of tools (created via @tool) to decide which actions to take. 
"""

In [11]:
resp=model_with_tools.invoke("whats the weather in hyderbad")
for tool_call in resp.tool_calls:
    #view tool calls made by model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'city': 'Hyderabad'}


### tool execution loops

In [ ]:
# s1: model genrates tool calls
messages=[{"role":"user","content":"whats the weather in kerala"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

# s2:execute tools and collect results
for tool_calls in ai_msg.tool_calls:
    tool_result=get_weather.invoke(tool_calls)
    messages.append(tool_result)

# s3:pass results back to moel for final response
final_response=model_with_tools.invoke(messages)
print(final_response)

content='The weather in Kerala is chilly!' additional_kwargs={'reasoning_content': 'The weather in Kerala is reported as chilly. I should inform the user about this.\n'} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 326, 'total_tokens': 354, 'completion_time': 0.055252406, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.02663161, 'prompt_tokens_details': None, 'queue_time': 0.04635705, 'total_time': 0.081884016}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_60e0d2f248', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fc242-2d79-7bf2-856a-826f873f1e43-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 326, 'output_tokens': 28, 'total_tokens': 354, 'output_token_details': {'reasoning': 18}}


Step 1: Model Generates Tool Calls

messages=[{"role":"user","content":"..."}]: Creates the starting conversation with your question.

ai_msg=model_with_tools.invoke(messages): Sends the question to the AI. The AI stops and returns a "plan" (a tool call) instead of an answer because it doesn't know the weather itself. 

messages.append(ai_msg): Saves the AI's plan into the conversation history so it remembers what it asked for. 


Step 2: Execute Tools (The Loop)

for tool_calls in ai_msg.tool_calls:: Starts a loop to handle every tool the AI requested (it might ask for multiple things at once).

tool_result=get_weather.invoke(tool_calls): This is the most important line. It actually runs your Python function get_weather using the arguments the AI chose.  This gets the real data (e.g., "Chilly!").

messages.append(tool_result): Saves the result ("Chilly!") back into the conversation history as a "Tool Message". 


Step 3: Final Response

final_response=model_with_tools.invoke(messages): Sends the updated history (Question + AI Plan + Tool Result) back to the AI.

print(final_response): The AI now sees the weather data and generates a natural sentence answer like "The weather in Kerala is chilly!" 

Why the loop? The loop is the bridge. The AI can ask for the weather, but only your Python code (inside the loop) can fetch it. The loop fetches the data and hands it back to the AI.

### MESSAGE

In [14]:
#one way
from langchain.messages import SystemMessage,HumanMessage
messages=[
    SystemMessage(content="u r a helpful assistant"),
    HumanMessage(content="what is 2+2")
]

model.invoke(messages)

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "what is 2+2".\n2.  **Identify Core Task:** Simple arithmetic addition.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely. "2 + 2 equals 4."\n5.  **Check for Tone/Constraints:** The prompt says "u r a helpful assistant". Keep it straightforward and helpful.\n6.  **Final Output Generation:** "2 + 2 equals 4." (or similar)✅\n</think>\n\n2 + 2 equals 4. Let me know if you need help with anything else!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 26, 'total_tokens': 184, 'completion_time': 0.302551115, 'completion_tokens_details': None, 'prompt_time': 0.00160679, 'prompt_tokens_details': None, 'queue_time': 0.05306696, 'total_time': 0.304157905}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_3a782ca0f8', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'lo

In [16]:
#another way like detail info to system message
sys_mesg=SystemMessage(content="""
    youa re a helpfu ai asistant who is an expert in maths and statistics.
    you have a noble proze as the ebst mathematician ever lived!,
    you answer things in a minute
""")

messages=[
    sys_mesg,
    HumanMessage(content="whats bays theorem and what naive in it be brief")
]

resp=model.invoke(messages)
resp.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Question:** "whats bays theorem and what naive in it be brief"\n   - **Key Concepts:** Bayes\' Theorem, Naive Bayes\n   - **Constraints:** "be brief"\n   - **Persona:** Helpful AI assistant, expert in math/stats, "noble prize as the best mathematician ever lived!", answers in a minute (implies concise, accurate, direct)\n\n2.  **Deconstruct Core Concepts:**\n   - **Bayes\' Theorem:** A fundamental rule in probability theory that updates the probability of a hypothesis as more evidence becomes available. Formula: P(A|B) = [P(B|A) * P(A)] / P(B)\n   - **Naive Bayes:** A classification algorithm based on Bayes\' Theorem with a "naive" assumption of feature independence. Used heavily in machine learning (e.g., spam filtering, text classification).\n\n3.  **Draft - Mental Refinement (Brief & Accurate):**\n   Bayes’ Theorem is a probability rule that updates the likelihood of an event based on new evidence:  \n   

In [17]:
resp.usage_metadata

{'input_tokens': 70, 'output_tokens': 928, 'total_tokens': 998}

### tool message

In [19]:
from langchain.messages import AIMessage,ToolMessage

#afte a model makes a tool call
#here we demostrate manually creating the message for brevity
ai_msg=AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args":{"city":"San Francisco"},
        "id":"call_123"
    }]
)

#execute tool and create resuklt
weather_result="SUnny, 97F"
tool_message=ToolMessage( #toolmessage represnts 'the output of the tool call!
    content=weather_result,
    tool_call_id="call_123"
)

#continue conversation
message=[
    HumanMessage("Whats the weather like in melbourne"),
    ai_msg,
    tool_message
]

response=model.invoke(messages)

In [21]:
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Topic:** Bayes\' Theorem\n   - **Specific Question:** What is Bayes\' Theorem? What does "naive" mean in it? (Referring to Naive Bayes)\n   - **Constraints:** Be brief, answer in a minute (implies concise, direct, efficient)\n   - **Persona:** Helpful AI assistant, expert in math/stats, noble prize winner (best mathematician ever), answers quickly\n\n2.  **Identify Key Concepts:**\n   - **Bayes\' Theorem:** Formula for updating probabilities based on new evidence.\n     - P(A|B) = [P(B|A) * P(A)] / P(B)\n     - Components: Posterior, Likelihood, Prior, Marginal/Evidence\n   - **"Naive" in Naive Bayes:** Refers to the assumption of conditional independence between features given the class label.\n   - **Briefness:** Keep it to a few sentences, clear and direct.\n\n3.  **Draft - Mental Refinement:**\n   Bayes’ Theorem updates a prior belief with new evidence:  \n   P(A|B) = [P(B|A) × P(A)] / 